# 🌿 GreenNetwork: Energy-Aware SDN Routing with Deep RL

**Train a DQN agent with adaptive clustering to optimize energy consumption in Software-Defined Networks**

---

## 📋 Overview

This notebook trains a Deep Q-Network (DQN) agent to:
- **Minimize energy consumption** by intelligently deactivating network links
- **Maintain QoS constraints** (latency, SLA requirements)
- **Adapt to traffic patterns** using dynamic clustering (DP-means)

### Key Features:
- ✅ Hierarchical action space with per-cluster control
- ✅ Adaptive network clustering
- ✅ Real-time visualization
- ✅ GPU acceleration
- ✅ Google Drive integration for model persistence

---

## 1️⃣ Setup & Installation

In [ ]:
# Check if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✅ Running in Google Colab")
    
    # Clone repository (if not already cloned)
    import os
    if not os.path.exists('Research-GreenNetwork'):
        print("📥 Cloning repository...")
        !git clone https://github.com/YOUR_USERNAME/Research-GreenNetwork.git
        %cd Research-GreenNetwork/colab-training
    else:
        print("✅ Repository already cloned")
        %cd Research-GreenNetwork/colab-training
else:
    print("⚠️  Not running in Colab - make sure you're in the colab-training directory")

In [ ]:
# Install dependencies
print("📦 Installing dependencies...")
!pip install -q -r requirements.txt
print("✅ Dependencies installed!")

In [ ]:
# Import libraries
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import pandas as pd
import os
import sys

# Add src to path
sys.path.insert(0, './src')

from env import SDNEnv
from agent import HierarchicalDQN
from utils import (
    ColabVisualizer, 
    setup_colab_environment, 
    mount_google_drive,
    print_training_header,
    print_episode_summary
)

print("✅ All imports successful!")

## 2️⃣ Environment Setup

In [ ]:
# Setup Colab environment
device, in_colab = setup_colab_environment()

# Mount Google Drive (optional)
SAVE_TO_DRIVE = True  # Set to False to save locally

if SAVE_TO_DRIVE and in_colab:
    save_dir = mount_google_drive()
else:
    save_dir = './models'
    os.makedirs(save_dir, exist_ok=True)
    print(f"✅ Models will be saved to: {save_dir}")

## 3️⃣ Configuration

Choose a pre-configured scenario or customize your own:

In [ ]:
# Choose configuration
# Options: 'tiny_network', 'small_network', 'medium_network', 'large_network'
CONFIG_NAME = 'small_network'  # 👈 Change this to select different network sizes

# Load configuration
config_path = f'configs/{CONFIG_NAME}.json'
with open(config_path, 'r') as f:
    config = json.load(f)['config']

# Override device
config['device'] = device

# Print configuration
print_training_header(config)

print("\n📊 Network Scenarios:")
print("  - tiny_network:   20 nodes,  40 edges (2-5 min)")
print("  - small_network:  50 nodes, 100 edges (5-10 min)")
print("  - medium_network: 100 nodes, 500 edges (15-30 min)")
print("  - large_network:  200 nodes, 2000 edges (1-2 hours)")

### 🎛️ Custom Configuration (Optional)

Uncomment and modify to customize training:

In [ ]:
# Uncomment to customize
# config['episodes'] = 1000
# config['lr'] = 0.001
# config['traffic_load_mode'] = 'high'  # 'low' or 'high'
# config['no_clustering'] = False  # Set True to disable clustering
# config['clustering_method'] = 'dp_means_adaptive'  # or 'silhouette', 'elbow', etc.

print("✅ Configuration ready!")

## 4️⃣ Initialize Environment & Agent

In [ ]:
# Initialize environment
print("🌐 Initializing SDN environment...")
env = SDNEnv(config)
obs = env.reset()

print(f"✅ Environment created:")
print(f"   Observation dim: {obs.shape[0]}")
print(f"   Action space: {env.action_n}")
print(f"   Network: {env.G_full.number_of_nodes()} nodes, {env.G_full.number_of_edges()} edges")

# Initialize agent
print("\n🤖 Initializing DQN agent...")
agent = HierarchicalDQN(
    obs_dim=obs.shape[0],
    action_n=env.action_n,
    cfg=config,
    device=device
)

print(f"✅ Agent created:")
print(f"   Q-Network parameters: {sum(p.numel() for p in agent.q.parameters()):,}")
print(f"   Device: {device}")
print(f"   Replay buffer size: {config['buffer_size']:,}")

# Initialize visualizer
visualizer = ColabVisualizer(update_every=10)
print("\n✅ Visualizer ready!")

## 5️⃣ Training Loop

**This cell will train the agent. You can interrupt it anytime (Runtime → Interrupt execution)**

In [ ]:
# Training loop
print("\n🎬 Starting training...\n")

best_reward = float('-inf')
episode_rewards = []
t_global = 0

# Progress bar
pbar = tqdm(range(config['episodes']), desc="Training", ncols=100)

try:
    for episode in pbar:
        obs = env.reset()
        total_reward = 0.0
        step_losses = []
        
        # Episode loop
        for step in range(config['max_steps_per_episode']):
            # Agent action
            action = agent.act(obs)
            
            # Environment step
            obs_next, reward, done, info = env.step(action)
            
            # Store experience
            agent.push(obs, action, reward, obs_next, float(done))
            obs = obs_next
            total_reward += reward
            
            # Training
            if len(agent.rb) >= config['batch_size'] and t_global % config['train_every'] == 0:
                loss = agent.train_step()
                if loss is not None:
                    step_losses.append(loss)
            
            # Update target network
            if t_global % config['target_update'] == 0:
                agent.update_target()
            
            t_global += 1
            
            if done:
                break
        
        # Episode metrics
        avg_loss = np.mean(step_losses) if step_losses else 0.0
        energy_saving = info.get('energy_saving', 0.0)
        latency = info.get('latency_ms', 0.0)
        sla_viol = info.get('sla_viol', 0.0)
        active_links = info.get('active_links', 0)
        
        util_stats = env.get_current_utilization_stats()
        utilization = util_stats['average_utilization']
        
        clustering_stats = env.get_clustering_statistics()
        cluster_count = clustering_stats['current_cluster_count']
        
        # Log metrics
        visualizer.log(
            episode=episode + 1,
            reward=total_reward,
            loss=avg_loss,
            energy_saving=energy_saving,
            latency=latency,
            sla_viol=sla_viol,
            active_links=active_links,
            utilization=utilization,
            cluster_count=cluster_count
        )
        
        episode_rewards.append(total_reward)
        
        # Save best model
        if total_reward > best_reward:
            best_reward = total_reward
            model_path = os.path.join(save_dir, f'best_model_{CONFIG_NAME}.pth')
            agent.save(model_path)
        
        # Periodic checkpoint
        if (episode + 1) % config.get('save_every', 100) == 0:
            checkpoint_path = os.path.join(save_dir, f'checkpoint_{CONFIG_NAME}_ep{episode+1}.pth')
            agent.save(checkpoint_path)
        
        # Update progress bar
        pbar.set_postfix({
            'Reward': f'{total_reward:.1f}',
            'Energy': f'{energy_saving*100:.1f}%',
            'SLA': f'{sla_viol:.1f}%',
            'ε': f'{agent.eps:.3f}'
        })
        
        # Live visualization
        visualizer.plot_live(episode + 1)

except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")

print("\n✅ Training completed!")

## 6️⃣ Results & Analysis

In [ ]:
# Display summary statistics
print("\n📊 Training Summary:\n")
summary = visualizer.get_summary()
display(summary)

# Save metrics to CSV
csv_path = os.path.join(save_dir, f'metrics_{CONFIG_NAME}.csv')
visualizer.save_to_csv(csv_path)

In [ ]:
# Interactive Plotly dashboard
print("📈 Interactive Dashboard:\n")
visualizer.plot_interactive()

In [ ]:
# Final clustering analysis
clustering_stats = env.get_clustering_statistics()

print("\n🔗 Clustering Analysis:")
print(f"   Method: {clustering_stats['clustering_method_used']}")
print(f"   Final cluster count: {clustering_stats['current_cluster_count']}")
print(f"   Average cluster count: {clustering_stats['avg_cluster_count']:.1f}")
print(f"   Cluster count std: {clustering_stats['cluster_count_std']:.2f}")
print(f"   Reclustering events: {clustering_stats['reclustering_events']}")
print(f"   Nodes per cluster: {clustering_stats['nodes_per_cluster']:.1f}")

## 7️⃣ Model Evaluation

Test the trained model:

In [ ]:
# Load best model
best_model_path = os.path.join(save_dir, f'best_model_{CONFIG_NAME}.pth')
agent.load(best_model_path, map_location=device)
print(f"✅ Loaded best model from: {best_model_path}")

# Evaluation
print("\n🧪 Evaluating model...\n")

eval_episodes = config.get('eval_episodes', 5)
eval_rewards = []
eval_energy_savings = []
eval_sla_violations = []

for ep in range(eval_episodes):
    obs = env.reset()
    total_reward = 0.0
    
    for step in range(config['max_steps_per_episode']):
        # Greedy action (no exploration)
        with torch.no_grad():
            q_values = agent.q(torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0))
            action = int(q_values.argmax(dim=1).item())
        
        obs, reward, done, info = env.step(action)
        total_reward += reward
        
        if done:
            break
    
    eval_rewards.append(total_reward)
    eval_energy_savings.append(info.get('energy_saving', 0.0) * 100)
    eval_sla_violations.append(info.get('sla_viol', 0.0))
    
    print(f"Episode {ep+1}/{eval_episodes}: Reward={total_reward:.2f}, "
          f"Energy={info.get('energy_saving', 0.0)*100:.1f}%, "
          f"SLA={info.get('sla_viol', 0.0):.1f}%")

print("\n📊 Evaluation Results:")
print(f"   Average Reward: {np.mean(eval_rewards):.2f} ± {np.std(eval_rewards):.2f}")
print(f"   Average Energy Saving: {np.mean(eval_energy_savings):.1f}% ± {np.std(eval_energy_savings):.1f}%")
print(f"   Average SLA Violations: {np.mean(eval_sla_violations):.1f}% ± {np.std(eval_sla_violations):.1f}%")

## 8️⃣ Export Results

Download results to your local machine:

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # Download metrics CSV
    csv_path = os.path.join(save_dir, f'metrics_{CONFIG_NAME}.csv')
    if os.path.exists(csv_path):
        files.download(csv_path)
        print(f"✅ Downloaded: {csv_path}")
    
    # Download best model
    model_path = os.path.join(save_dir, f'best_model_{CONFIG_NAME}.pth')
    if os.path.exists(model_path):
        files.download(model_path)
        print(f"✅ Downloaded: {model_path}")
else:
    print("✅ Results saved locally")
    print(f"   Metrics: {csv_path}")
    print(f"   Model: {model_path}")

---

## 🎉 Training Complete!

### Next Steps:

1. **Try different configurations**: Change `CONFIG_NAME` to test different network sizes
2. **Experiment with traffic modes**: Set `config['traffic_load_mode'] = 'high'`
3. **Disable clustering**: Set `config['no_clustering'] = True` to see the impact
4. **Adjust hyperparameters**: Modify learning rate, epsilon decay, etc.
5. **Compare with baselines**: Check the `experiment/` folder for baseline implementations

### 📚 Resources:

- [GitHub Repository](https://github.com/YOUR_USERNAME/Research-GreenNetwork)
- [Documentation](../README.md)
- [Paper References](../train/experiment/README.md)

---

**Happy Training! 🚀**